# ProjectMem Kaggle Full Runner

Notebook nay chay full benchmark suite bang `app/main.py` truc tiep tren Kaggle.

- Agent chinh: `Qwen/Qwen2.5-7B-Instruct`
- Agent doi chieu: `microsoft/Phi-4-mini-instruct`
- Co do GPU, progress artifact, ETA theo benchmark, va fallback ve structured/dummy khi run model that bai
- Runtime se fail fast neu GPU thuc te khong phai `T4`
- Output duoc luu trong `/kaggle/working/projectmem_reports_full`


In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/Dung205789/MultiAgentConflictResolution.git"
BRANCH = "main"
WORKDIR = "/kaggle/working/ProjectMem"

if os.path.isdir(WORKDIR):
    subprocess.run(["git", "-C", WORKDIR, "fetch", "origin"], check=False)
    subprocess.run(["git", "-C", WORKDIR, "checkout", BRANCH], check=False)
    subprocess.run(["git", "-C", WORKDIR, "pull", "origin", BRANCH], check=False)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, WORKDIR], check=True)

os.chdir(WORKDIR)
print("Working directory:", os.getcwd())
print(subprocess.check_output(["git", "rev-parse", "--short", "HEAD"]).decode().strip())


In [ ]:
%cd /kaggle/working/ProjectMem
!python -m pip install -q --upgrade pip
!python -m pip install -q -r requirements.txt

import os
from huggingface_hub import login

hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    hf_token = os.environ.get("HF_TOKEN")

if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print("Hugging Face login complete.")
else:
    print("HF_TOKEN not set. Public benchmarks and models can still download if available.")


In [ ]:
import json
import os
import subprocess
import time
from pathlib import Path

import torch

USE_DUMMY = False
ALLOW_DUMMY_FALLBACK = True
MAX_SCENARIOS = None
DEVICE = "auto"
OUTPUT_ROOT = "/kaggle/working/projectmem_reports_full"
PROGRESS_PATH = f"{OUTPUT_ROOT}/progress.json"
GPU_PROFILE_PATH = f"{OUTPUT_ROOT}/gpu_profile.json"

MODEL_NAME1 = "Qwen/Qwen2.5-7B-Instruct"
MODEL_NAME2 = "microsoft/Phi-4-mini-instruct"
PRIMARY_AGENT_ROLE = "primary_extractor"
SECONDARY_AGENT_ROLE = "challenger_extractor"

REQUIRE_GPU = True
REQUIRE_GPU_SUBSTRING = "T4"

INCLUDE_LOCAL_MEMAE = False
MEMAE_LOCAL_PATH = "data/raw/memab/Conflict_Resolution-00000-of-00001.parquet"

BENCHMARK_SPECS = [
    {"benchmark": "mab_conflict"},
    {"benchmark": "mab", "subset": "all"},
    {"benchmark": "longmemeval", "subset": "oracle"},
    {"benchmark": "safeflow"},
    {"benchmark": "locomo", "subset": "all"}
]

if INCLUDE_LOCAL_MEMAE:
    BENCHMARK_SPECS.insert(0, {"benchmark": "memae"})

os.makedirs(OUTPUT_ROOT, exist_ok=True)

def run_text_command(cmd):
    completed = subprocess.run(cmd, check=False, capture_output=True, text=True)
    return {
        "command": cmd,
        "returncode": completed.returncode,
        "stdout": completed.stdout.strip(),
        "stderr": completed.stderr.strip(),
    }

def measure_gpu_profile():
    profile = {
        "cuda_available": torch.cuda.is_available(),
        "required_gpu_substring": REQUIRE_GPU_SUBSTRING,
    }
    if not torch.cuda.is_available():
        return profile

    props = torch.cuda.get_device_properties(0)
    gpu_name = torch.cuda.get_device_name(0)
    profile.update({
        "gpu_name": gpu_name,
        "total_memory_gb": round(props.total_memory / (1024 ** 3), 2),
        "multi_processor_count": props.multi_processor_count,
        "cuda_capability": f"{props.major}.{props.minor}",
    })

    smi = run_text_command(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"])
    profile["nvidia_smi"] = smi

    torch.cuda.empty_cache()
    size = 4096
    a = torch.randn((size, size), device="cuda", dtype=torch.float16)
    b = torch.randn((size, size), device="cuda", dtype=torch.float16)
    for _ in range(3):
        _ = a @ b
    torch.cuda.synchronize()
    started = time.perf_counter()
    iters = 8
    for _ in range(iters):
        _ = a @ b
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - started
    avg_seconds = elapsed / iters
    flops = 2 * (size ** 3)
    tflops = flops / avg_seconds / 1e12
    profile["matmul_benchmark"] = {
        "matrix_size": size,
        "dtype": "float16",
        "iterations": iters,
        "avg_seconds": round(avg_seconds, 4),
        "estimated_tflops": round(tflops, 2),
    }
    return profile

gpu_profile = measure_gpu_profile()
Path(GPU_PROFILE_PATH).write_text(json.dumps(gpu_profile, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(gpu_profile, ensure_ascii=False, indent=2))

if REQUIRE_GPU and not gpu_profile.get("cuda_available"):
    raise RuntimeError("Notebook requires a CUDA GPU, but torch.cuda.is_available() is false.")

gpu_name = gpu_profile.get("gpu_name", "")
if REQUIRE_GPU_SUBSTRING and REQUIRE_GPU_SUBSTRING.lower() not in gpu_name.lower():
    raise RuntimeError(f"Expected a GPU containing {REQUIRE_GPU_SUBSTRING!r}, but Kaggle provided {gpu_name!r}.")

print("Agent 1 model:", MODEL_NAME1, PRIMARY_AGENT_ROLE)
print("Agent 2 model:", MODEL_NAME2, SECONDARY_AGENT_ROLE)
print("Allow dummy fallback:", ALLOW_DUMMY_FALLBACK)
print("Benchmark specs:", BENCHMARK_SPECS)


In [ ]:
import json
import shlex
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

from tqdm.auto import tqdm

output_root = Path(OUTPUT_ROOT)
progress_path = Path(PROGRESS_PATH)
summary = {
    "timestamp": datetime.now(timezone.utc).isoformat().replace("+00:00", "Z"),
    "benchmarks": [],
    "use_dummy": USE_DUMMY,
    "device": DEVICE,
    "agent1_model": None if USE_DUMMY else MODEL_NAME1,
    "agent2_model": None if USE_DUMMY else MODEL_NAME2,
    "agent_roles": {
        "agent1": PRIMARY_AGENT_ROLE,
        "agent2": SECONDARY_AGENT_ROLE,
    },
    "max_scenarios": MAX_SCENARIOS,
    "allow_dummy_fallback": ALLOW_DUMMY_FALLBACK,
    "gpu_profile": gpu_profile,
}

def write_progress(current_index, current_run_name, status, benchmark_results, started_at, current_started_at=None):
    now = time.time()
    completed = len([x for x in benchmark_results if x.get("status") in {"success", "failed"}])
    total = len(BENCHMARK_SPECS)
    completed_durations = [x.get("elapsed_seconds") for x in benchmark_results if x.get("elapsed_seconds") is not None]
    avg_seconds = sum(completed_durations) / len(completed_durations) if completed_durations else None
    remaining = total - completed
    eta_seconds = None
    if avg_seconds is not None and remaining > 0:
        eta_seconds = round(avg_seconds * remaining, 2)
    progress = {
        "timestamp": datetime.now(timezone.utc).isoformat().replace("+00:00", "Z"),
        "status": status,
        "current_index": current_index,
        "total_benchmarks": total,
        "current_run_name": current_run_name,
        "completed_benchmarks": completed,
        "remaining_benchmarks": remaining,
        "elapsed_total_seconds": round(now - started_at, 2),
        "elapsed_current_seconds": None if current_started_at is None else round(now - current_started_at, 2),
        "average_completed_benchmark_seconds": None if avg_seconds is None else round(avg_seconds, 2),
        "estimated_remaining_seconds": eta_seconds,
        "benchmark_results": benchmark_results,
        "gpu_name": gpu_profile.get("gpu_name"),
        "gpu_tflops_estimate": gpu_profile.get("matmul_benchmark", {}).get("estimated_tflops"),
    }
    progress_path.write_text(json.dumps(progress, ensure_ascii=False, indent=2), encoding="utf-8")
    return progress

if INCLUDE_LOCAL_MEMAE and not Path(MEMAE_LOCAL_PATH).exists():
    print(f"WARNING: {MEMAE_LOCAL_PATH} not found. memae run will fail unless you add that parquet first.")

benchmark_results = []
session_started = time.time()
pbar = tqdm(total=len(BENCHMARK_SPECS), desc="ProjectMem Kaggle Benchmarks", unit="benchmark")
write_progress(0, None, "starting", benchmark_results, session_started)

for idx, spec in enumerate(BENCHMARK_SPECS, start=1):
    benchmark = spec["benchmark"]
    run_name = benchmark if "subset" not in spec else f"{benchmark}_{spec['subset']}"
    output_dir = output_root / run_name
    output_dir.mkdir(parents=True, exist_ok=True)
    current_started = time.time()
    pbar.set_description(f"Running {run_name}")
    write_progress(idx - 1, run_name, "running", benchmark_results, session_started, current_started)

    cmd = [
        sys.executable,
        "app/main.py",
        "--benchmark", benchmark,
        "--output-dir", str(output_dir),
        "--device", DEVICE,
    ]

    if MAX_SCENARIOS is not None:
        cmd.extend(["--max-scenarios", str(MAX_SCENARIOS)])
    if "subset" in spec:
        cmd.extend(["--subset", spec["subset"]])

    if USE_DUMMY:
        cmd.append("--use-dummy")
    else:
        cmd.extend([
            "--agent1-model", MODEL_NAME1,
            "--agent2-model", MODEL_NAME2,
        ])

    print("\n" + "=" * 72)
    print(f"Running benchmark via app/main.py: {run_name}")
    print(shlex.join(cmd))
    print("=" * 72)

    completed = subprocess.run(cmd, check=False)
    fallback_completed = None

    if completed.returncode != 0 and not USE_DUMMY and ALLOW_DUMMY_FALLBACK:
        fallback_cmd = [
            sys.executable,
            "app/main.py",
            "--benchmark", benchmark,
            "--output-dir", str(output_dir),
            "--device", DEVICE,
            "--use-dummy",
        ]
        if MAX_SCENARIOS is not None:
            fallback_cmd.extend(["--max-scenarios", str(MAX_SCENARIOS)])
        if "subset" in spec:
            fallback_cmd.extend(["--subset", spec["subset"]])

        print("\n" + "-" * 72)
        print(f"Primary run failed for {run_name}. Retrying in dummy mode.")
        print(shlex.join(fallback_cmd))
        print("-" * 72)
        fallback_completed = subprocess.run(fallback_cmd, check=False)

    effective = fallback_completed or completed
    elapsed_seconds = round(time.time() - current_started, 2)
    result = {
        "benchmark": benchmark,
        "run_name": run_name,
        "subset": spec.get("subset"),
        "returncode": effective.returncode,
        "status": "success" if effective.returncode == 0 else "failed",
        "primary_returncode": completed.returncode,
        "used_dummy_fallback": fallback_completed is not None,
        "fallback_returncode": None if fallback_completed is None else fallback_completed.returncode,
        "output_dir": str(output_dir),
        "elapsed_seconds": elapsed_seconds,
    }
    benchmark_results.append(result)
    summary["benchmarks"].append(result)
    pbar.update(1)
    progress = write_progress(idx, run_name, result["status"], benchmark_results, session_started)
    print(f"Completed {run_name} in {elapsed_seconds}s. ETA remaining: {progress.get('estimated_remaining_seconds')}")

pbar.close()
summary_path = output_root / "matrix_summary.json"
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
write_progress(len(BENCHMARK_SPECS), None, "complete", benchmark_results, session_started)
print("Saved summary:", summary_path)


In [ ]:
import zipfile
from pathlib import Path

output_root = Path(OUTPUT_ROOT)
for path in sorted(output_root.rglob("*.json")):
    print(path)

zip_path = Path("/kaggle/working/projectmem_kaggle_full_reports.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for file_path in output_root.rglob("*"):
        if file_path.is_file():
            archive.write(file_path, file_path.relative_to(output_root.parent))

print("Created:", zip_path)
